# Smart Microscopy v3 - Adaptive Target Acquisition

Two-objective workflow: acquire an overview scan at the source objective,
segment cells with cellpose, discover interesting targets in the operator-
inspectable distribution, then re-acquire those targets at the target
objective.

## Workflow

| Step | What happens | Operator action before running |
|------|-------------|-------------------------------|
| 1. Initialization | Connect LAS X, calibration, engine boot, slot derivation | Connect LAS X |
| 2. Setup overview | Configure spatial extent of the overview scan | (see substeps) |
| 2a. Define limits | Derive XY limits from markers, enforce z-wide | Place boundary markers in Navigator Expert |
| 2b. Define positions | Read tile positions from template | Draw scan field in Navigator Expert |
| 2c. Define focus map | AF at markers, fit z-wide surface model | Place focus markers on scan field |
| 3. Acquire overview | Snake-acquire all tiles, segment + persist | - |
| 4. Target discovery | Inspect cell distribution, apply thresholds, pick targets | - |
| 5. Target acquisition | Acquire each picked cell at the target objective | - |
| 6. Finish | Write `run_summary.json`, plot results, restore, shutdown | - |
| -- Cleanup | Safe shutdown after any failure | - |

All logic lives in `notebooks/workflow/`. This notebook is a thin
operator-facing wrapper. See `docs/TARGET_ACQUISITION_DESIGN.md` for the
full design.

## Configuration

Edit the cell below to match your experiment. Everything else runs
without modification.

**Operating modes** (set `analysis_image_source`):

| Mode | Value | Use case |
|------|-------|----------|
| Real microscope | `"acquired"` | Production: analyse the actual acquired tiles |
| Simulator / dry run | `"skimage_human_mitosis"` | Testing: use a standard cell image for every tile |

The mock mode can also be used on the real microscope intentionally —
it acquires real tiles but analyses a known reference image, useful for
validating the coordinate chain without depending on sample quality.

**Before a real microscope run:**

1. Set `analysis_image_source = "acquired"`
2. Verify objectives are configured correctly in LAS X for each job
3. Set z-galvo to 0 in LAS X (preflight will warn if non-zero)
4. Place boundary markers and focus markers before the respective steps

In [ ]:
from _workflow_bootstrap import Config, Path

cfg = Config(
    # Jobs (each has an objective configured in LAS X)
    acquisition_job="Overview",
    target_job="HiRes",
    af_job="AF Job",

    # Paths
    analysis_repo=Path(r"Z:\zmbstaff\10374\Protocols_Notes\thom\notes\repositories\smart-analysis"),
    experiment="v3-test",   # driver derives output_root as media_path/smart

    # Analysis image source. Pick one:
    #   "acquired"               - production: analyse the actual acquired tiles
    #   "skimage_human_mitosis"  - dry run: analyse a standard reference cell image
    analysis_image_source="acquired",
    # analysis_image_source="skimage_human_mitosis",
)

## Step 1 - Initialization

Connects to LAS X, validates hardware and calibration, derives objective
slots from the configured jobs, boots the analysis engine, and
establishes a deterministic starting state.

In [ ]:
from workflow import connect_lasx, preflight

client = connect_lasx()
ctx = preflight(cfg, client)


## Step 2 - Setup overview

Configure the spatial extent of the overview scan: stage limits, tile
positions, and a focus map. Run **2a → 2b → 2c** in order.

### Step 2a - Define limits

Optionally place **point markers** in Navigator Expert at the corners of your
sample. The cell will derive XY stage limits from those markers (with
`cfg.limit_margin_um` padding), clamped to the physical envelope from
`stage.json` (any clamp is reported per axis).

If no markers are placed, the physical envelope is applied immediately
so any intermediate move stays bounded; XY narrowing is then deferred
to Step 2b (derived from the scan field, also clamped to the envelope).

The cell also strips the template and forces every job in the LRP to **z-wide**.

In [ ]:
from workflow import prepare_template, plot_stage_envelope

prepare_template(ctx)
plot_stage_envelope(ctx)

### Step 2b - Define positions

Draw the scanning field in **Navigator Expert**, then run the cell below.
The scan field defines which tiles are acquired during overview.

In [ ]:
from workflow import read_scan_field, plot_scan_field

read_scan_field(ctx)
plot_scan_field(ctx)

### Step 2c - Define focus map

Place **focus markers** on the scan field in **Navigator Expert**, then
run this cell. Runs the AF job at each marker, reads back z-wide, and
fits a surface model (constant, line, plane, or thin plate spline
depending on the number of markers).

In [ ]:
from workflow import build_focus_map

focus_map = build_focus_map(ctx)
focus_map.plot(ctx)

## Step 3 - Acquire overview

Acquires every tile in snake order, submits each to the analysis engine
for cellpose segmentation, and persists each tile's metrics + Pick
reconstruction arrays to disk (NPZ schema v2 + `overview_meta.json`).
No selection happens here — that's Step 4.

Each tile shows a 3-panel figure inline:
**Field position** | **Tile image** | **Segmentation**.
The leftmost panel shows the full scan field with the *current tile
highlighted in red* (so you can see acquisition progress at a glance);
the middle panel is the grayscale tile; the right panel overlays
cellpose masks on the tile with the cell count in the title.

In [ ]:
from workflow import run_overview

overview = run_overview(ctx, focus_map)


## Step 4 - Target discovery

Loads overview state from disk — kernel-restart safe between Step 3 and
Step 4: re-running this cell in a fresh kernel works as long as the
`overview-scan/` directory is intact. Computes global median thresholds
(or applies operator overrides), samples up to `n_per_tile` qualifying
cells per tile, dedups, and filters out-of-limits picks.

**Border-margin filter**: cells whose bounding box falls within
`border_margin_px` of any tile edge are excluded from qualifying — they
have truncated area/intensity statistics and produce unreliable picks.
The default of `64` px works for most tile sizes; set to `0` to disable
(useful if your tiles overlap enough that edge cells are still whole,
or if you specifically want to study edge cells). Auto-thresholds are
computed on non-border cells only.

The scatter plot has 5 color categories:
**Near edge** (amber — bbox within `border_margin_px` of any tile edge) ·
**Below threshold** (gray — fails area/intensity) ·
**Qualifying** (blue — passed threshold but lost to dedup or out-of-limits) ·
**Selected** (navy — in final picks) ·
**Shown** (red — the cells in the crop strip below the scatter).

Re-run this cell with different `area_threshold` / `intensity_threshold`
/ `n_per_tile` / `border_margin_px` to iterate without re-acquiring
overview.

In [ ]:
from workflow.selection import select_targets, load_overview_result
from workflow.visualize import display_selection

_analysis_dir = ctx.run.layout.analysis_dir("overview-scan")
overview = load_overview_result(_analysis_dir)   # kernel-restart safe

# Auto thresholds by default; uncomment to override. border_margin_px=64
# excludes cells whose bbox is within 64 px of any tile edge -- they have
# truncated stats and produce unreliable picks. Set to 0 to disable.
picks, selection = select_targets(
    overview,
    ctx.limits_context(),
    n_per_tile=4,
    border_margin_px=64,
    # area_threshold=200,
    # intensity_threshold=100,
    # seed=42,
)
display_selection(
    selection, _analysis_dir,
    feedback_dir=ctx.run.layout.feedback_dir("overview-scan"),
)

## Step 5 - Target acquisition

Switches to the target job and acquires each picked cell at high
magnification. Per-pick failure isolation: if one pick fails, the
rest still acquire.

Each target shows a 3-panel figure inline: overview tile with marker |
centroid crop at target FOV | acquired high-res image.

In [ ]:
from workflow import acquire_targets

records = acquire_targets(ctx, picks)


## Step 6 - Finish

Writes `run_summary.json`, plots the results overview, restores the
source job, and shuts down the engine.

The **cleanup** cell below is a safety backstop for interrupted runs
— it is not part of the normal path.

In [ ]:
from workflow import write_summary, plot_results, finish
from workflow.selection import load_overview_result

# Reload from disk so the finish cell is consistent with selection above
# (restart-between-selection-and-finish is NOT supported -- picks,
# selection, and records must still be in-kernel here).
overview = load_overview_result(ctx.run.layout.analysis_dir("overview-scan"))
write_summary(ctx, focus_map, overview, picks, selection, records)
plot_results(ctx, focus_map, picks, records)
finish(ctx)

In [ ]:
# Cleanup -- safety backstop for interrupted runs. Shuts down the
# analysis engine. Does NOT disconnect the LAS X client; it persists
# until kernel restart. Always safe to run.
try:
    ctx.shutdown()
except NameError:
    pass
